In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import csv
from tqdm import tqdm

In [2]:
# #!/usr/bin/env python3
# """
# Federal Reserve Speeches Web Scraper - Selenium Version
# For sites with JavaScript-based pagination
# """

# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.chrome.options import Options
# from selenium.common.exceptions import TimeoutException, NoSuchElementException
# from bs4 import BeautifulSoup
# import time
# import csv
# import json


# class FedSpeechScraperSelenium:
#     def __init__(self, headless=True):
#         """
#         Initialize the Selenium-based scraper
        
#         Args:
#             headless: Run browser in headless mode (no GUI)
#         """
#         self.base_url = "https://www.federalreserve.gov/newsevents/speeches-testimony.htm"
#         self.all_speeches = []
        
#         # Setup Chrome options
#         chrome_options = Options()
#         if headless:
#             chrome_options.add_argument('--headless')
#         chrome_options.add_argument('--no-sandbox')
#         chrome_options.add_argument('--disable-dev-shm-usage')
#         chrome_options.add_argument('--disable-gpu')
#         chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
#         # Initialize the driver
#         try:
#             self.driver = webdriver.Chrome(options=chrome_options)
#         except Exception as e:
#             print(f"Error initializing Chrome driver: {e}")
#             print("Make sure ChromeDriver is installed and in your PATH")
#             raise
        
#         self.wait = WebDriverWait(self.driver, 10)
    
#     def parse_speeches_from_page(self):
#         """
#         Parse speeches from the current page
#         """
#         speeches = []
        
#         # Get page source and parse with BeautifulSoup
#         soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        
#         # Find the main content div
#         content_div = soup.find('div', class_='col-xs-12 col-sm-8 col-md-8', id='speech-results')
        
#         if not content_div:
#             print("Warning: Could not find the speech results div")
#             return speeches
        
#         # Find all speech rows
#         speech_rows = content_div.find_all('div', class_='row')
        
#         for row in speech_rows:
#             # Skip the pagination row
#             if row.find('nav', attrs={'aria-label': 'Page navigation'}):
#                 continue
                
#             # Extract date
#             date_elem = row.find('time', class_='itemDate')
#             if not date_elem:
#                 continue
                
#             date = date_elem.get('datetime', '')
            
#             # Extract title and link
#             title_elem = row.find('a', href=True)
#             if not title_elem:
#                 continue
                
#             title = title_elem.get_text(strip=True)
#             link = title_elem.get('href', '')
            
#             # Make the link absolute if it's relative
#             if link and not link.startswith('http'):
#                 link = f"https://www.federalreserve.gov{link}"
            
#             # Extract speaker
#             speaker_elem = row.find('p', class_='news__speaker')
#             speaker = speaker_elem.get_text(strip=True) if speaker_elem else ''
            
#             # Extract location
#             location_elem = row.find('p', class_='result__location')
#             location = location_elem.get_text(strip=True) if location_elem else ''
            
#             # Check if there's a watch live link
#             watch_live_elem = row.find('a', class_='watchLive')
#             watch_live = watch_live_elem.get('href', '') if watch_live_elem else ''
            
#             speech_data = {
#                 'date': date,
#                 'title': title,
#                 'link': link,
#                 'speaker': speaker,
#                 'location': location,
#                 'watch_live': watch_live
#             }
            
#             speeches.append(speech_data)
        
#         return speeches
    
#     def get_current_page_number(self):
#         """
#         Get the current page number from the active pagination button
#         """
#         try:
#             active_button = self.driver.find_element(By.CSS_SELECTOR, 'nav[aria-label="Page navigation"] li.active button')
#             aria_label = active_button.get_attribute('aria-label')
#             if 'Page' in aria_label:
#                 return int(aria_label.split('Page')[-1].strip())
#         except:
#             pass
#         return 1
    
#     def click_next_page(self):
#         """
#         Click the 'Next' button to go to the next page
#         Returns True if successful, False if no next page exists
#         """
#         try:
#             # Find the Next button
#             next_button = self.wait.until(
#                 EC.element_to_be_clickable((By.XPATH, "//nav[@aria-label='Page navigation']//button[@aria-label='Next']"))
#             )
            
#             # Check if the button is disabled
#             parent_li = next_button.find_element(By.XPATH, '..')
#             if 'disabled' in parent_li.get_attribute('class'):
#                 return False
            
#             # Click the button
#             next_button.click()
            
#             # Wait for the page to load
#             time.sleep(2)
            
#             return True
#         except (TimeoutException, NoSuchElementException):
#             return False
    
#     def scrape_all_pages(self, max_pages=27):
#         """
#         Scrape all pages using Selenium
#         """
#         print("Starting Federal Reserve Speech Scraper (Selenium)...")
#         print(f"Target: {max_pages} pages")
#         print("-" * 60)
        
#         try:
#             # Load the first page
#             self.driver.get(self.base_url)
#             time.sleep(3)  # Wait for page to load
            
#             page_count = 0
            
#             while page_count < max_pages:
#                 page_count += 1
#                 current_page = self.get_current_page_number()
                
#                 print(f"Scraping page {current_page}...")
                
#                 # Parse speeches from current page
#                 speeches = self.parse_speeches_from_page()
#                 self.all_speeches.extend(speeches)
                
#                 print(f"Page {current_page}: Found {len(speeches)} speeches")
                
#                 # Try to go to next page
#                 if page_count < max_pages:
#                     if not self.click_next_page():
#                         print(f"No more pages available. Stopped at page {current_page}")
#                         break
                    
#                     # Small delay between pages
#                     time.sleep(2)
            
#             print("-" * 60)
#             print(f"Scraping complete! Total speeches collected: {len(self.all_speeches)}")
            
#         finally:
#             # Always close the driver
#             self.driver.quit()
    
#     def save_to_csv(self, filename='fed_speeches_selenium.csv'):
#         """
#         Save collected speeches to a CSV file
#         """
#         if not self.all_speeches:
#             print("No speeches to save")
#             return
        
#         with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
#             fieldnames = ['date', 'title', 'speaker', 'location', 'link', 'watch_live']
#             writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
#             writer.writeheader()
#             for speech in self.all_speeches:
#                 writer.writerow(speech)
        
#         print(f"\nData saved to {filename}")
    
#     def save_to_json(self, filename='fed_speeches_selenium.json'):
#         """
#         Save collected speeches to a JSON file
#         """
#         if not self.all_speeches:
#             print("No speeches to save")
#             return
        
#         with open(filename, 'w', encoding='utf-8') as jsonfile:
#             json.dump(self.all_speeches, jsonfile, indent=2, ensure_ascii=False)
        
#         print(f"Data saved to {filename}")


# def main():
#     """
#     Main function to run the Selenium scraper
#     """
#     scraper = FedSpeechScraperSelenium(headless=True)
    
#     try:
#         # Scrape all 27 pages
#         scraper.scrape_all_pages(max_pages=27)
        
#         # Save results
#         scraper.save_to_csv('dataset/new_fed_speech_link.csv')
        
#         # Display statistics
#         if scraper.all_speeches:
#             print("\n" + "=" * 60)
#             print("SUMMARY STATISTICS")
#             print("=" * 60)
#             print(f"Total speeches scraped: {len(scraper.all_speeches)}")
            
#             dates = [s['date'] for s in scraper.all_speeches if s['date']]
#             if dates:
#                 print(f"Date range: {min(dates)} to {max(dates)}")
            
#             print("\n" + "-" * 60)
#             print("Sample of first 3 speeches:")
#             print("-" * 60)
#             for i, speech in enumerate(scraper.all_speeches[:3], 1):
#                 print(f"\n{i}. {speech['title']}")
#                 print(f"   Date: {speech['date']}")
#                 print(f"   Speaker: {speech['speaker']}")
#                 print(f"   Link: {speech['link']}")
    
#     except KeyboardInterrupt:
#         print("\n\nScraping interrupted by user")
#     except Exception as e:
#         print(f"\n\nError during scraping: {e}")
#         raise


# if __name__ == "__main__":
#     main()

In [3]:
# import pandas as pd

# df = pd.read_csv("dataset/new_fed_speech_link.csv")
# df = df['link']
# df.to_csv("dataset/new_fed_speech_link.csv", index=False)

Scrap Each Link

In [4]:
# # ==========================
# # TEXT CLEANING
# # ==========================

# def clean_text(text):
#     if not text:
#         return ""

#     text = text.replace(",", "")
#     text = text.replace('"', "")
#     text = text.replace("\n", " ")
#     text = re.sub(r"\s+", " ", text)

#     return text.strip()


# # ==========================
# # NEWSEVENTS SCRAPER
# # ==========================

# def scrape_newsevents(soup):

#     title = ""
#     date = ""
#     speaker = ""
#     content = ""

#     # Title
#     title_tag = soup.find("h3", class_="title")
#     if title_tag:
#         title = clean_text(title_tag.get_text())

#     # Date
#     date_tag = soup.find("p", class_="article__time")
#     if date_tag:
#         date = clean_text(date_tag.get_text())

#     # Speaker
#     speaker_tag = soup.find("p", class_="speaker")
#     if speaker_tag:
#         speaker = clean_text(speaker_tag.get_text())

#     # Content
#     content_div = soup.find("div", class_="col-xs-12 col-sm-8 col-md-8")
#     if content_div:
#         paragraphs = content_div.find_all("p")
#         content = clean_text(
#             " ".join(
#                 p.get_text()
#                 for p in paragraphs
#                 if p.get_text().strip() != ""
#             )
#         )

#     return title, date, speaker, content


# # ==========================
# # MAIN PIPELINE
# # ==========================

# df = pd.read_csv("dataset/new_fed_speech_link.csv")

# results = []

# for link in tqdm(df["link"]):

#     try:
#         response = requests.get(link, timeout=15)
#         soup = BeautifulSoup(response.text, "html.parser")

#         if "newsevents" in link:
#             title, date, speaker, content = scrape_newsevents(soup)

#         results.append({
#             "link": link,
#             "title": title,
#             "date": date,
#             "speaker": speaker,
#             "content": content
#         })

#     except Exception as e:
#         print(f"Error scraping {link}: {e}")
#         results.append({
#             "link": link,
#             "title": "",
#             "date": "",
#             "speaker": "",
#             "content": ""
#         })


# output_df = pd.DataFrame(results)

# output_df.to_csv(
#     "dataset/new_fed_speech_content.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL
# )

# print("Scraping complete.")


In [14]:
df = pd.read_csv("dataset/new_fed_speech_content.csv")
df['date'] = pd.to_datetime(df['date'], format='%B %d %Y')
df = df.sort_values(by='date')
df = df.drop(columns=['link'])
df = df[df['date'] > '2020-06-19']
df['id'] = range(1456, 1456 + len(df))

df = df[['id', 'date', 'title', 'speaker', 'content']]
df.to_csv('dataset/new_fed_speech_content_unquoted.csv', index=False)